# 위치 인코딩 RoPE/ALiBi 실습 - RoPE Relative Position Property

- Tutorial ID: `mod-rope-alibi-lab`
- Tutorial: 위치 인코딩 RoPE/ALiBi 실습
- Section ID: `pos-1`
- Section: RoPE Relative Position Property


In [ ]:
# ============================================================
# 코드 읽는 법 — RoPE Relative Position Property
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 위치 정보가 벡터 회전/편향으로 attention score에 들어가는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
def rot(x, pos, theta=0.2):
    c, s = np.cos(pos*theta), np.sin(pos*theta)
    R = np.array([[c,-s],[s,c]])
    return R @ x
q = np.array([1.0, 0.3]); k = np.array([0.2, 1.1])
for i,j in [(2,5),(10,13),(1,4)]:
    print((i,j),'delta',j-i,'dot=',round(rot(q,i) @ rot(k,j), 6))
print('same delta gives same dot up to numerical precision')
# ALiBi: distance penalty
T=6; slope=-0.4
dist = np.arange(T)[None,:] - np.arange(T)[:,None]
alibi = slope*np.maximum(dist,0)
print('
ALiBi future/long-distance bias:
', alibi)